# 03a Setup Fixation

**Phase 3.1 - Apply Setup Decisions to Create ML-Ready Datasets**

This notebook applies the finalized setup decisions from exploratory notebooks (exp_08-exp_09) to create filtered datasets for Phase 3 experiments.

**Dependencies:** Requires `setup_decisions.json` from exp_09 (complete pipeline).

**Output:** Creates 5 filtered parquet files (berlin_train, berlin_val, berlin_test, leipzig_finetune, leipzig_test) with applied ablation decisions.

---

**RUNTIME REQUIREMENTS:**
- CPU: Standard (12GB RAM sufficient)
- GPU: Not required
- High-RAM: Optional (recommended for comfort)

**IMPORTANT:** Test splits always use baseline proximity (no filtering) to represent real-world distribution for unbiased evaluation.

In [ ]:
# ============================================================
# INSTALLATION
# ============================================================
# SETUP: Add GITHUB_TOKEN to Colab Secrets (key icon in sidebar)
# ============================================================

import subprocess
from google.colab import userdata

# Get GitHub token from Colab Secrets
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets.\n"
        "1. Click the key icon in the left sidebar\n"
        "2. Add a secret named 'GITHUB_TOKEN' with your GitHub token\n"
        "3. Toggle 'Notebook access' ON"
    )

# Install package from GitHub
repo_url = f"git+https://{token}@github.com/SilasPignotti/urban-tree-transfer.git"
subprocess.run(["pip", "install", repo_url, "-q"], check=True)

print("✅ Package installed successfully")

In [ ]:
# ============================================================
# GOOGLE DRIVE MOUNT
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("✅ Google Drive mounted")

In [ ]:
# ============================================================
# IMPORTS & INITIALIZATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings
import gc

import pandas as pd
import numpy as np

from urban_tree_transfer.config import load_experiment_config, RANDOM_SEED
from urban_tree_transfer.experiments import ablation
from urban_tree_transfer.utils import (
    ExecutionLog,
    setup_plotting,
    validate_setup_decisions,
)

setup_plotting()
warnings.filterwarnings("ignore", category=UserWarning)

log = ExecutionLog("03a_setup_fixation")

print("✅ Imports complete")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DRIVE_DIR = Path("/content/drive/MyDrive/dev/urban-tree-transfer")
DATA_DIR = DRIVE_DIR / "data"
INPUT_DIR = DATA_DIR / "phase_2_splits"
OUTPUT_DIR = DATA_DIR / "phase_3_experiments"

METADATA_DIR = OUTPUT_DIR / "metadata"
LOGS_DIR = OUTPUT_DIR / "logs"
FIGURES_DIR = OUTPUT_DIR / "figures"

# Create output directories
for d in [OUTPUT_DIR, METADATA_DIR, LOGS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Load experiment configuration
config = load_experiment_config()

print(f"Input (Phase 2 Splits):  {INPUT_DIR}")
print(f"Output (Phase 3):        {OUTPUT_DIR}")
print(f"Metadata:                {METADATA_DIR}")
print(f"Logs:                    {LOGS_DIR}")
print(f"Random seed:             {RANDOM_SEED}")
print(f"\n✅ Configuration complete")

In [ ]:
# ============================================================
# SECTION 1: Load Setup Decisions
# ============================================================

log.start_step("Load Setup Decisions")

try:
    setup_path = METADATA_DIR / "setup_decisions.json"
    
    if not setup_path.exists():
        raise FileNotFoundError(
            f"setup_decisions.json not found at {setup_path}\n"
            "Run exploratory notebooks exp_08 → exp_08b → exp_08c → exp_09 first."
        )
    
    # Validate and load setup decisions
    setup = validate_setup_decisions(setup_path)
    
    print("\n" + "=" * 70)
    print("Setup Decisions Loaded")
    print("=" * 70)
    print(f"CHM Strategy:       {setup['chm_strategy']['decision']}")
    print(f"Proximity Strategy: {setup['proximity_strategy']['decision']}")
    print(f"Outlier Strategy:   {setup['outlier_strategy']['decision']}")
    print(f"Selected Features:  {len(setup['selected_features'])}")
    print("=" * 70)
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 2: Apply Decisions to Create Filtered Datasets
# ============================================================

log.start_step("Apply Setup Decisions")

try:
    # Define all city-split combinations
    city_splits = [
        ("berlin", "train"),
        ("berlin", "val"),
        ("berlin", "test"),
        ("leipzig", "finetune"),
        ("leipzig", "test"),
    ]
    
    results = []
    
    print("\n" + "=" * 70)
    print("Processing Splits")
    print("=" * 70)
    
    for city, split in city_splits:
        split_name = f"{city}_{split}"
        print(f"\n[{split_name}]")
        
        # Apply setup decisions (test split policy enforced automatically)
        df, meta = ablation.prepare_ablation_dataset(
            base_path=INPUT_DIR,
            city=city,
            split=split,
            setup_decisions=setup,
            return_metadata=True,
            optimize_memory=True,
        )
        
        # Save filtered dataset
        output_path = OUTPUT_DIR / f"{split_name}.parquet"
        df.to_parquet(output_path, index=False)
        
        # Track results
        results.append({
            "split": split_name,
            "n_samples": len(df),
            "n_features": meta["n_features"],
            "outliers_removed": meta["outliers_removed"],
            "test_policy_applied": meta["test_split_policy_applied"],
            "proximity_used": meta["proximity_strategy_used"],
        })
        
        # Report
        print(f"  Samples: {meta['original_n_samples']:,} → {len(df):,}")
        print(f"  Features: {meta['n_features']}")
        print(f"  Outliers removed: {meta['outliers_removed']}")
        if meta["test_split_policy_applied"]:
            print(f"  ⚠️  Test split policy: proximity override baseline")
        print(f"  ✅ Saved: {output_path.name}")
        
        # Memory cleanup
        del df, meta
        gc.collect()
    
    # Summary
    results_df = pd.DataFrame(results)
    print("\n" + "=" * 70)
    print("Summary")
    print("=" * 70)
    print(results_df.to_string(index=False))
    print("=" * 70)
    
    log.end_step(status="success", records=len(results))
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 3: Validation
# ============================================================

log.start_step("Validation")

try:
    print("\n" + "=" * 70)
    print("Validation Checks")
    print("=" * 70)
    
    validation_passed = True
    
    # Check all output files exist
    for city, split in city_splits:
        split_name = f"{city}_{split}"
        output_path = OUTPUT_DIR / f"{split_name}.parquet"
        
        if not output_path.exists():
            print(f"❌ Missing: {split_name}.parquet")
            validation_passed = False
            continue
        
        # Load and validate
        df = pd.read_parquet(output_path)
        
        # Check feature count matches
        feature_cols = [c for c in df.columns if c not in ablation.get_metadata_columns(df)]
        if len(feature_cols) != len(setup["selected_features"]):
            print(f"❌ {split_name}: Feature count mismatch ({len(feature_cols)} != {len(setup['selected_features'])})")
            validation_passed = False
        
        # Check for NaN values in features
        nan_count = df[feature_cols].isna().sum().sum()
        if nan_count > 0:
            print(f"❌ {split_name}: Contains {nan_count} NaN values")
            validation_passed = False
        
        # Check sample count is reasonable
        if len(df) < 100:
            print(f"⚠️  {split_name}: Very few samples ({len(df)})")
        
        print(f"✅ {split_name}: {len(df):,} samples, {len(feature_cols)} features, 0 NaN")
    
    print("=" * 70)
    
    if validation_passed:
        print("\n✅ All validation checks passed")
        log.end_step(status="success")
    else:
        print("\n❌ Validation failed")
        log.end_step(status="error", errors=["Validation checks failed"])
        raise ValueError("Validation checks failed")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SECTION 4: Save Execution Log
# ============================================================

log.start_step("Save Execution Log")

try:
    # Add summary metadata
    execution_summary = {
        "notebook": "03a_setup_fixation",
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "setup_decisions": {
            "chm": setup["chm_strategy"]["decision"],
            "proximity": setup["proximity_strategy"]["decision"],
            "outlier": setup["outlier_strategy"]["decision"],
            "n_features": len(setup["selected_features"]),
        },
        "test_split_policy": "Test splits always use baseline proximity (no filtering)",
        "splits_processed": len(results),
        "total_samples": sum(r["n_samples"] for r in results),
        "splits": results,
    }
    
    # Save execution log
    log.summary()
    log_path = LOGS_DIR / f"{log.notebook}_execution.json"
    log.save(log_path)
    
    # Save summary
    summary_path = METADATA_DIR / "03a_summary.json"
    summary_path.write_text(json.dumps(execution_summary, indent=2))
    
    print(f"\n✅ Execution log saved: {log_path}")
    print(f"✅ Summary saved: {summary_path}")
    
    log.end_step(status="success")
    
except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise

In [ ]:
# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("NOTEBOOK COMPLETE: 03a Setup Fixation")
print("=" * 70)
print(f"\nSetup Decisions Applied:")
print(f"  CHM:       {setup['chm_strategy']['decision']}")
print(f"  Proximity: {setup['proximity_strategy']['decision']}")
print(f"  Outlier:   {setup['outlier_strategy']['decision']}")
print(f"  Features:  {len(setup['selected_features'])}")
print(f"\nSplits Processed: {len(results)}")
print(f"Total Samples:    {sum(r['n_samples'] for r in results):,}")
print(f"\nOutputs:")
print(f"  Datasets: {OUTPUT_DIR}")
for r in results:
    print(f"    - {r['split']}.parquet ({r['n_samples']:,} samples)")
print(f"  Logs:     {log_path}")
print(f"  Summary:  {summary_path}")
print(f"\nTest Split Policy:")
print(f"  ℹ️  Test splits use baseline proximity (no filtering)")
print(f"  ℹ️  Ensures unbiased evaluation on real-world distribution")
print(f"\nNext Steps:")
print(f"  1. Review filtered datasets")
print(f"  2. Run exp_10_algorithm_comparison.ipynb (if not done)")
print(f"  3. Run 03b_berlin_optimization.ipynb (hyperparameter tuning)")
print("=" * 70)